In [9]:
import pandas as pd
import numpy as np
import re
import os

In [10]:
print("Current folder:")
print(os.getcwd())

print("\nFiles in current folder:")
print(os.listdir())

Current folder:
c:\Users\HP\Downloads\University_Ranking_Deliverables

Files in current folder:
['data_collection.py', 'education_cleaning.ipynb', 'university_cleaned.csv', 'university_raw_data.csv']


In [11]:
RAW_FILE = "university_raw_data.csv"

raw = pd.read_csv(RAW_FILE)

print("Raw data loaded successfully")
print("Shape:", raw.shape)

Raw data loaded successfully
Shape: (3008, 31)


In [12]:
df = raw.copy()

print("Working copy created")
print("Rows:", len(df))
print("Columns:", len(df.columns))

Working copy created
Rows: 3008
Columns: 31


In [13]:
def clean_col(c):
    return re.sub(
        r'[^A-Za-z0-9]+',
        '_',
        str(c).strip()
    ).strip('_')


df.columns = [clean_col(c) for c in df.columns]

print("Cleaned column names:")
print(df.columns.tolist())

Cleaned column names:
['Rank', 'Previous_Rank', 'Name', 'Country_Territory', 'Region', 'Size', 'Focus', 'Research', 'Status', 'Academic_Reputation_SCORE', 'Data_Source', 'Academic_Reputation_RANK', 'Employer_Reputation_SCORE', 'Employer_Reputation_RANK', 'Faculty_Student_Ratio_SCORE', 'Faculty_Student_Ratio_RANK', 'Citations_per_Faculty_SCORE', 'Citations_per_Faculty_RANK', 'International_Faculty_SCORE', 'International_Faculty_RANK', 'International_Student_SCORE', 'International_student_RANK', 'International_Students_Diversity_SCORE', 'International_Students_Diversity_RANK', 'International_Research_Network_SCORE', 'International_Research_Network_RANK', 'Employment_Outcomes_SCORE', 'Employment_Outcomes_RANK', 'Sustainability_SCORE', 'Sustainability_RANK', 'Overall_SCORE']


In [14]:
def clean_text(x):
    if pd.isna(x):
        return np.nan
    
    x = str(x).strip()
    x = re.sub(r'\s+', ' ', x)
    
    return x if x else np.nan


for c in df.select_dtypes(include='object').columns:
    df[c] = df[c].map(clean_text)

print("Text cleaning completed")

C:\Users\HP\AppData\Local\Temp\ipykernel_13520\2937492481.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for c in df.select_dtypes(include='object').columns:


Text cleaning completed


In [15]:
missing = df.isna().sum()

print("Missing values:")
print(missing[missing > 0].sort_values(ascending=False))

Missing values:
International_Faculty_RANK                1591
International_Faculty_SCORE               1591
International_student_RANK                1541
International_Student_SCORE               1541
International_Students_Diversity_RANK     1541
International_Students_Diversity_SCORE    1541
Sustainability_RANK                       1528
Sustainability_SCORE                      1528
International_Research_Network_RANK       1506
International_Research_Network_SCORE      1506
Citations_per_Faculty_RANK                1504
Faculty_Student_Ratio_RANK                1504
Academic_Reputation_RANK                  1504
Employer_Reputation_SCORE                 1504
Employer_Reputation_RANK                  1504
Faculty_Student_Ratio_SCORE               1504
Citations_per_Faculty_SCORE               1504
Employment_Outcomes_SCORE                 1504
Employment_Outcomes_RANK                  1504
Overall_SCORE                             1504
Previous_Rank                              2

In [16]:
missing_percent = (
    df.isna().mean() * 100
).sort_values(ascending=False)

print("Missing percentage:")
print(missing_percent[missing_percent > 0])

Missing percentage:
International_Faculty_SCORE               52.892287
International_Faculty_RANK                52.892287
International_Student_SCORE               51.230053
International_student_RANK                51.230053
International_Students_Diversity_SCORE    51.230053
International_Students_Diversity_RANK     51.230053
Sustainability_SCORE                      50.797872
Sustainability_RANK                       50.797872
International_Research_Network_RANK       50.066489
International_Research_Network_SCORE      50.066489
Employer_Reputation_SCORE                 50.000000
Academic_Reputation_RANK                  50.000000
Citations_per_Faculty_RANK                50.000000
Employer_Reputation_RANK                  50.000000
Faculty_Student_Ratio_SCORE               50.000000
Citations_per_Faculty_SCORE               50.000000
Employment_Outcomes_SCORE                 50.000000
Employment_Outcomes_RANK                  50.000000
Faculty_Student_Ratio_RANK                50

In [17]:
print(df.columns.tolist())

['Rank', 'Previous_Rank', 'Name', 'Country_Territory', 'Region', 'Size', 'Focus', 'Research', 'Status', 'Academic_Reputation_SCORE', 'Data_Source', 'Academic_Reputation_RANK', 'Employer_Reputation_SCORE', 'Employer_Reputation_RANK', 'Faculty_Student_Ratio_SCORE', 'Faculty_Student_Ratio_RANK', 'Citations_per_Faculty_SCORE', 'Citations_per_Faculty_RANK', 'International_Faculty_SCORE', 'International_Faculty_RANK', 'International_Student_SCORE', 'International_student_RANK', 'International_Students_Diversity_SCORE', 'International_Students_Diversity_RANK', 'International_Research_Network_SCORE', 'International_Research_Network_RANK', 'Employment_Outcomes_SCORE', 'Employment_Outcomes_RANK', 'Sustainability_SCORE', 'Sustainability_RANK', 'Overall_SCORE']


In [18]:
print("Duplicate university names:", df['Name'].duplicated().sum())

Duplicate university names: 1504


In [19]:
df = df.drop_duplicates()

print("Duplicates removed")
print("New shape:", df.shape)

Duplicates removed
New shape: (3008, 31)


In [20]:
def rank_num(x):
    if pd.isna(x):
        return np.nan

    s = str(x).strip()

    # Standardize dash characters
    s = (
        s.replace('–', '-')
         .replace('—', '-')
         .replace('=', '')
    )

    # Find numbers
    nums = re.findall(
        r'\d+(?:\.\d+)?',
        s
    )

    if not nums:
        return np.nan

    values = [float(v) for v in nums]

    # If ranking is a range such as 101-150,
    # use the average
    if '-' in s and len(values) >= 2:
        return sum(values) / len(values)

    return values[0]

In [21]:
for col in df.columns:
    if 'rank' in col.lower():
        print(col)

Rank
Previous_Rank
Academic_Reputation_RANK
Employer_Reputation_RANK
Faculty_Student_Ratio_RANK
Citations_per_Faculty_RANK
International_Faculty_RANK
International_student_RANK
International_Students_Diversity_RANK
International_Research_Network_RANK
Employment_Outcomes_RANK
Sustainability_RANK


In [22]:
if 'Rank' in df.columns:
    df['Rank_Number'] = df['Rank'].map(rank_num)

print(df[['Rank', 'Rank_Number']].head(10))

  Rank  Rank_Number
0    1          1.0
1    2          2.0
2    3          3.0
3    4          4.0
4    5          5.0
5    6          6.0
6    7          7.0
7    8          8.0
8    9          9.0
9   10         10.0


In [23]:
if 'Rank_QS' in df.columns:
    df['QS_Rank_Number'] = df['Rank_QS'].map(rank_num)

if 'Rank_World' in df.columns:
    df['World_Rank_Number'] = df['Rank_World'].map(rank_num)

In [24]:
def normalize(s):
    s = pd.to_numeric(s, errors='coerce')

    lo = s.min()
    hi = s.max()

    if pd.isna(lo):
        return pd.Series(
            np.nan,
            index=s.index
        )

    if lo == hi:
        return pd.Series(
            100.0,
            index=s.index
        )

    return (
        (hi - s) /
        (hi - lo) *
        100
    ).round(2)

In [25]:
df['Rank_Normalized'] = normalize(
    df['Rank_Number']
)

print(
    df[
        ['Rank', 'Rank_Number', 'Rank_Normalized']
    ].head(10)
)

  Rank  Rank_Number  Rank_Normalized
0    1          1.0           100.00
1    2          2.0            99.93
2    3          3.0            99.86
3    4          4.0            99.79
4    5          5.0            99.71
5    6          6.0            99.64
6    7          7.0            99.57
7    8          8.0            99.50
8    9          9.0            99.43
9   10         10.0            99.36


In [26]:
print("Final shape:", df.shape)

print("\nFirst 5 rows:")
display(df.head())

print("\nData types:")
print(df.dtypes)

Final shape: (3008, 33)

First 5 rows:


,Rank,Previous_Rank,Name,Country_Territory,Region,Size,Focus,Research,Status,Academic_Reputation_SCORE,...,International_Students_Diversity_RANK,International_Research_Network_SCORE,International_Research_Network_RANK,Employment_Outcomes_SCORE,Employment_Outcomes_RANK,Sustainability_SCORE,Sustainability_RANK,Overall_SCORE,Rank_Number,Rank_Normalized
0,1,1,Massachusetts Institute of Technology (MIT),United States of America,Americas,M,CO,VH,Private not for Profit,100.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,100.00
1,2,2,Imperial College London,United Kingdom,Europe,L,FO,VH,Public,99.6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,99.93
2,3,6,Stanford University,United States of America,Americas,L,FC,VH,Private not for Profit,100.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,99.86
3,4,3,University of Oxford,United Kingdom,Europe,L,FC,VH,Public,100.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,99.79
4,5,4,Harvard University,United States of America,Americas,L,FC,VH,Private not for Profit,100.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,99.71



Data types:
Rank                                          str
Previous_Rank                                 str
Name                                          str
Country_Territory                             str
Region                                        str
Size                                          str
Focus                                         str
Research                                      str
Status                                        str
Academic_Reputation_SCORE                 float64
Data_Source                                   str
Academic_Reputation_RANK                  float64
Employer_Reputation_SCORE                 float64
Employer_Reputation_RANK                  float64
Faculty_Student_Ratio_SCORE               float64
Faculty_Student_Ratio_RANK                float64
Citations_per_Faculty_SCORE               float64
Citations_per_Faculty_RANK                float64
International_Faculty_SCORE               float64
International_Faculty_RANK           

In [27]:
for c in df.select_dtypes(
    include=np.number
).columns:

    median = df[c].median()

    if pd.notna(median):
        df[c] = df[c].fillna(median)

print("Numerical missing values handled")

Numerical missing values handled


In [28]:
for c in df.select_dtypes(
    include='object'
).columns:

    mode = df[c].mode(dropna=True)

    if len(mode) > 0:
        df[c] = df[c].fillna(mode.iloc[0])

print("Text missing values handled")

Text missing values handled


C:\Users\HP\AppData\Local\Temp\ipykernel_13520\855126315.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for c in df.select_dtypes(


In [29]:
df = df.drop_duplicates().reset_index(drop=True)

print("Duplicates removed")
print("Final rows:", len(df))

Duplicates removed
Final rows: 3008


In [30]:
completeness = (
    100 -
    df.isna().mean().mean() * 100
)

print(
    "Data completeness:",
    round(completeness, 2),
    "%"
)

Data completeness: 100.0 %


In [31]:
OUTPUT_FILE = "university_cleaned_final.csv"

df.to_csv(
    OUTPUT_FILE,
    index=False
)

print(
    f"Saved successfully: {OUTPUT_FILE}"
)

Saved successfully: university_cleaned_final.csv


In [32]:
check = pd.read_csv(
    "university_cleaned_final.csv"
)

print("Saved file shape:", check.shape)
print(check.head())

Saved file shape: (3008, 33)
  Rank Previous_Rank                                         Name  \
0    1             1  Massachusetts Institute of Technology (MIT)   
1    2             2                      Imperial College London   
2    3             6                          Stanford University   
3    4             3                         University of Oxford   
4    5             4                           Harvard University   

          Country_Territory    Region Size Focus Research  \
0  United States of America  Americas    M    CO       VH   
1            United Kingdom    Europe    L    FO       VH   
2  United States of America  Americas    L    FC       VH   
3            United Kingdom    Europe    L    FC       VH   
4  United States of America  Americas    L    FC       VH   

                   Status  Academic_Reputation_SCORE  ...  \
0  Private not for Profit                      100.0  ...   
1                  Public                       99.6  ...   
2  Pri